# Hybrid ALNS Performance Evaluation

## Section 1 - Introduction

Bin packing is a combinatorial optimization problem. We are given a list of item weights and identical bins with a fixed capacity. Every item must be assigned to exactly one bin, the total weight in each bin must not exceed the capacity, and the objective is to minimize the number of bins used.

This problem is difficult because a good packing depends on many interacting assignment decisions. A placement that looks good for one item can block better placements later. For realistic instance sizes, exact methods can become expensive, so heuristic and metaheuristic solvers are commonly used.

The solver evaluated here is a hybrid **Adaptive Large Neighborhood Search** solver, abbreviated **ALNS**. It starts from a feasible packing, repeatedly destroys part of the current solution, repairs the removed items, and keeps track of the best packing found.

The word **large** matters. A small local search move might move one item or swap two items. ALNS removes a group of items and rebuilds that part of the solution, so one iteration can change many assignments at once. This lets the solver escape local patterns that are hard to fix with single-item moves.

The solver uses three destroy operators:

- **Random destroy** removes a random set of items.
- **Worst-bin destroy** removes items from poorly filled bins.
- **Related-items destroy** removes items with similar sizes, giving repair a chance to reorganize a structured part of the packing.

After destruction, the solver repairs the partial solution. With learning disabled, repair uses a deterministic best-fit rule. With learning enabled, an offline Gradient Boosted Trees model scores feasible bin placements and selects the highest-scoring placement for each displaced item.

Candidate solutions are accepted with simulated annealing. If a candidate improves or preserves the bin count, it is accepted. If it is worse, it can still be accepted with probability `exp(-delta / temperature)`, where `delta` is the increase in bin count. The temperature cools over time, so the search is more exploratory early and more selective later.

The hybrid architecture has two learning components:

- **Offline GBT repair model**: trained before the benchmark and used to guide repair decisions.
- **Online LinUCB bandit**: updated during the ALNS run and used to choose the destroy operator from the current search context.

This notebook evaluates two questions:

1. In the ablation study, how much do the offline model and online bandit contribute separately and together?
2. In the multi-dataset benchmark, how does the combined approach behave across datasets with different sizes, capacities, and item distributions?

---

## Utilities Used in This Notebook

The notebook uses three project utilities instead of reimplementing benchmarking logic inside notebook cells.

**Benchmarking utility**

The benchmarking utility creates a benchmark from a dataset key and a solver module. It loads matching instances, runs the solver, records one result row per instance, and saves the result table as CSV. Each row includes the instance name, dataset key, item count, bin capacity, bins used, lower bound, total weight, elapsed time, method label, and timeout status.

**Statistics utility**

The statistics utility loads benchmark CSV files and computes summary metrics. The main quality metric is `gap = bins_used - lower_bound`. Lower gap is better. The lower bound comes from known benchmark metadata when available; otherwise it is the continuous weight bound `ceil(total_weight / bin_capacity)`. The utility also reports average bins used, solve time, completion count, and fill rate. Fill rate is `total_weight / (bins_used * bin_capacity)`, so higher fill rate usually means less wasted capacity.

**Graphing utility**

The graphing utility reads benchmark CSV files and creates standard figures. Per-dataset graphs show solve time, bins versus lower bound, fill rate, and gap versus time. Comparison graphs show solve time and average bins by method. Cross-dataset graphs show solve time distribution, average gap, and size-versus-time trends.

The utilities keep the notebook cells short and make the experiments reproducible.

This cell imports the solver and the three utilities. It also hides non-essential warnings caused by loading a model artifact saved with an older scikit-learn version.

In [ ]:
import warnings

from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
    hybrid_alns_solver,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.models import (
    load_repair_model,
)
from bin_packing_optimization.utilities.benchmarking import create_benchmark
import bin_packing_optimization.utilities.graphing as graphing
import bin_packing_optimization.utilities.statistics as statistics

warnings.filterwarnings("ignore", message="method=.*is ignored")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

---

## Section 2 - Ablation Study

The ablation uses **Scholl-2**, restricted to the smallest size class: 50 items and bin capacity 1000. Scholl instances are uniformly distributed benchmark instances with item counts from 50 to 500. Scholl-2 fixes the bin capacity at 1000.

The benchmark is limited to the first five sorted 50-item instances. This keeps the experiment short while still showing quality differences between learning configurations.

All four ablation runs use the same ALNS settings: 300 iterations, initial temperature `1 / log(2) = 1.4426950408889634`, and cooling rate `0.9995`. Only the two learning switches change.

### Configuration 1 - No Learning

This disables both learning components. Destroy operators are selected randomly and repair uses best-fit. This is the baseline.

In [ ]:
benchmark = create_benchmark("scholl-2", hybrid_alns_solver, time_limit=3.0)

benchmark.run(
    method="no-learning",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": False,
        "use_online_rl": False,
    },
    max_instances=5,
)

no_learning_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/ablation/no-learning.csv"
)

### Configuration 2 - Online Only

This keeps best-fit repair but enables the online LinUCB bandit for destroy-operator selection. The expected benefit is better destroy-operator choice than random selection.

In [ ]:
benchmark = create_benchmark("scholl-2", hybrid_alns_solver, time_limit=3.0)

benchmark.run(
    method="online-only",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": False,
        "use_online_rl": True,
    },
    max_instances=5,
)

online_only_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/ablation/online-only.csv"
)

### Configuration 3 - Offline Only

This enables the GBT repair model but keeps random destroy-operator selection. The expected benefit is better repair placement decisions, with additional runtime from model scoring.

In [ ]:
benchmark = create_benchmark("scholl-2", hybrid_alns_solver, time_limit=3.0)

benchmark.run(
    method="offline-only",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": False,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=5,
)

offline_only_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/ablation/offline-only.csv"
)

### Configuration 4 - Both Combined

This enables both learning components. The LinUCB bandit chooses destroy operators online and the GBT model guides repair. This is expected to improve average bins and average gap compared with the no-learning baseline.

In [ ]:
benchmark = create_benchmark("scholl-2", hybrid_alns_solver, time_limit=3.0)

benchmark.run(
    method="both-combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=5,
)

both_combined_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/ablation/both-combined.csv"
)

This cell prints a method comparison table. The important columns are average bins, average gap, average fill rate, and average time.

In [ ]:
statistics.print_method_comparison_report(
    [no_learning_csv, online_only_csv, offline_only_csv, both_combined_csv]
)

This cell creates the ablation graphs: solve-time distribution by method and average bins by method.

In [ ]:
graphing.display_comparison_graphs(
    [no_learning_csv, online_only_csv, offline_only_csv, both_combined_csv],
    out_dir="results/performance_evaluation/ablation/graphs",
)

### Ablation Summary

The ablation isolates the solver components. The no-learning configuration gives the reference result. The online-only run tests whether adaptive destroy selection helps when repair is still greedy. The offline-only run tests whether learned repair helps when destroy selection is random. The combined run tests the full hybrid architecture.

The main quality comparison is no-learning versus both-combined. A lower average gap and lower average bins in the combined run indicates that the learning components improve solution quality on this benchmark slice. Runtime should be interpreted separately because learned repair performs model scoring during repair.

---

## Section 3 - Multi-Dataset Benchmarking

The multi-dataset study uses the combined approach only. Both learning components are enabled for every dataset.

The algorithm settings are fixed across all datasets:

| Parameter | Value | Purpose |
|---|---:|---|
| `max_iterations` | 300 | Fixed evaluation budget for each instance in this notebook |
| `initial_temperature` | 1.4426950408889634 | Standard simulated annealing start temperature, equal to `1 / log(2)` |
| `alpha_cool` | 0.9995 | Standard cooling rate used after each ALNS iteration |
| `use_offline_model` | `True` | Enables the trained GBT repair model |
| `use_online_rl` | `True` | Enables online destroy-operator selection |
| Repair model | `repair_model_v2.pkl` | Fixed model artifact used for all datasets |

The LinUCB settings are the solver's fixed internal values: exploration parameter `alpha = 0.3` and Thompson-sampling warm-up of 300 bandit calls. The solver seed remains the default `42`, so stochastic choices are reproducible.

### Dataset 1 - Scholl-2

Scholl-2 contains uniformly distributed instances with fixed bin capacity 1000 and item counts from 50 to 500. This benchmark uses five 50-item instances. It provides a small uniform baseline with moderate capacity.

In [ ]:
benchmark = create_benchmark("scholl-2", hybrid_alns_solver, time_limit=3.0)

benchmark.run(
    method="combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=830,
)

scholl2_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/multi_dataset/scholl-2.csv"
)

This cell prints Scholl-2 statistics and displays the per-dataset graphs.

In [ ]:
statistics.print_benchmark_report(scholl2_csv)
graphing.display_graphs(scholl2_csv)

### Dataset 2 - Falkenauer-T

Falkenauer-T contains triplet-structured instances. In these instances, one large item and two small items often need to be packed together in an optimal solution. Capacity is 1000 and item counts range from 60 to 501. This benchmark uses five 60-item instances.

In [ ]:
benchmark = create_benchmark("falkenauer-t", hybrid_alns_solver, time_limit=5.0)

benchmark.run(
    method="combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=508,
)

falkenauer_t_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/multi_dataset/falkenauer-t.csv"
)

This cell prints Falkenauer-T statistics and displays the per-dataset graphs.

In [ ]:
statistics.print_benchmark_report(falkenauer_t_csv)
graphing.display_graphs(falkenauer_t_csv)

### Dataset 3 - Falkenauer-U

Falkenauer-U contains uniformly distributed item sizes, capacity 150, and item counts from 120 to 1000. This benchmark uses five 120-item instances. It adds more items and tighter capacity than the Scholl-2 slice.

In [ ]:
benchmark = create_benchmark("falkenauer-u", hybrid_alns_solver, time_limit=12.0)

benchmark.run(
    method="combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=141,
)

falkenauer_u_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/multi_dataset/falkenauer-u.csv"
)

This cell prints Falkenauer-U statistics and displays the per-dataset graphs.

In [ ]:
statistics.print_benchmark_report(falkenauer_u_csv)
graphing.display_graphs(falkenauer_u_csv)

### Dataset 4 - Wascher

Wascher contains difficult cutting-stock style benchmark instances. The bin capacity is 10000 and item counts vary across the set, from small instances to larger hard cases. This benchmark uses five sorted instances to include structural variation inside the same family.

In [ ]:
benchmark = create_benchmark("wäscher", hybrid_alns_solver, time_limit=6.0)

benchmark.run(
    method="combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=425,
)

wascher_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/multi_dataset/wascher.csv"
)

This cell prints Wascher statistics and displays the per-dataset graphs.

In [ ]:
statistics.print_benchmark_report(wascher_csv)
graphing.display_graphs(wascher_csv)

### Dataset 5 - Hard28

Hard28 contains difficult instances with 160 to 200 items and capacity 1000. This benchmark uses three sorted instances because this family is slower, while still representing the hard-instance setting.

In [ ]:
benchmark = create_benchmark("hard28", hybrid_alns_solver, time_limit=30.0)

benchmark.run(
    method="combined",
    method_args={
        "max_iterations": 300,
        "initial_temperature": 1.4426950408889634,
        "alpha_cool": 0.9995,
        "use_offline_model": True,
        "use_online_rl": True,
        "model_bundle": load_repair_model("repair_model_v2.pkl"),
    },
    max_instances=50,
)

hard28_csv = benchmark.save_results_to_csv(
    "results/performance_evaluation/multi_dataset/hard28.csv"
)

This cell prints Hard28 statistics and displays the per-dataset graphs.

In [ ]:
statistics.print_benchmark_report(hard28_csv)
graphing.display_graphs(hard28_csv)

### Cross-Dataset Aggregate Results

This subsection aggregates the five combined-mode CSV files. The statistics table compares datasets using the same fixed parameter setting. The graphs show global trends in runtime, gap, and scaling with item count.

In [ ]:
statistics.print_multi_benchmark_report(
    [scholl2_csv, falkenauer_t_csv, falkenauer_u_csv, wascher_csv, hard28_csv]
)

This cell creates and displays the aggregate cross-dataset graphs.

In [ ]:
graphing.display_multi_dataset_graphs(
    [scholl2_csv, falkenauer_t_csv, falkenauer_u_csv, wascher_csv, hard28_csv],
    out_dir="results/performance_evaluation/multi_dataset/graphs",
)